# EDA: inspeção inicial (registo MIT-BIH 100)

Este notebook carrega um registo WFDB a partir do extrato local **MIT-BIH** em `data/raw/`, imprime campos básicos do cabeçalho e traça os **primeiros 10 segundos** de ambos os canais de ECG. As amostras de anotação do ficheiro `atr` aparecem como linhas verticais de referência; nesta base alinham-se com etiquetas de batimento junto ao complexo QRS (uso frequente como instante de pico R em classificação de batimentos).

## Pré-requisito

Execute `python -m src.data.download_dataset` após a configuração (`pip install -r requirements.txt` e `pip install -e .`) para que existam ficheiros WFDB. A célula seguinte obtém o diretório do registo com `src.config.mitdb_record_dir()`.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wfdb

from src.config import mitdb_record_dir

# Identificador do registo WFDB (sem extensão .hea / .dat).
RECORD_NAME = "100"
record_dir = str(mitdb_record_dir())
record_dir

In [ ]:
# Caminho completo do registo (local); evita pn_dir com caminho de disco no Windows.
record_path = Path(record_dir) / RECORD_NAME
record_path_str = record_path.as_posix()
record = wfdb.rdrecord(record_path_str)
ann = wfdb.rdann(record_path_str, "atr")

## Metadados do cabeçalho

Frequência de amostragem `fs`, número de canais, nomes e duração do segmento em amostras.

In [ ]:
print("fs (Hz):", record.fs)
print("n_sig:", record.n_sig)
print("sig_name:", record.sig_name)
print("sig_len (amostras):", record.sig_len)
print("duração (s):", record.sig_len / record.fs)

## Traçado de dez segundos com marcadores de anotação

Extrai-se a janela dos primeiros `10 * fs` amostras e sobrepõem-se linhas verticais nos instantes de anotação dentro da janela. Os símbolos seguem as etiquetas de batimento MIT-BIH (por exemplo `N` normal, `V` ventricular, entre outras).

In [ ]:
fs = float(record.fs)
window = min(int(fs * 10), int(record.sig_len))
t = np.arange(window) / fs
sig = record.p_signal[:window, :]

fig, axes = plt.subplots(record.n_sig, 1, figsize=(12, 6), sharex=True)
if record.n_sig == 1:
    axes = [axes]

for i in range(record.n_sig):
    axes[i].plot(t, sig[:, i], color="tab:blue", linewidth=0.9)
    axes[i].set_ylabel(record.sig_name[i])
    axes[i].grid(True, alpha=0.3)

for sample in ann.sample:
    if sample >= window:
        continue
    x = sample / fs
    for ax in axes:
        ax.axvline(x, color="tab:red", alpha=0.35, linewidth=0.9)

axes[-1].set_xlabel("Tempo (s)")
fig.suptitle(
    f"MIT-BIH {RECORD_NAME}: primeiros 10 s (dois canais) com instantes de amostra atr"
)
fig.tight_layout()
plt.show()